In [11]:
# Importando bibliotecas
from faker import Faker
from random import choice, randint, uniform, choices
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from unidecode import unidecode
from hashlib import sha256
from openai import OpenAI

In [12]:
# Carregando variáveis de ambiente
load_dotenv()

# Acessando o BD principal e o anonimizado, e chave de api da hugging face
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_PRINC')
HUGGING_FACE = os.getenv('HUGGING_FACE')

# Criando engine
entrada = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

engine = create_engine(entrada)

# Carregando faker
f = Faker('pt_BR')

In [13]:
# Gerando observação sobre descarte de embalagem com Hugging Face
def gerar_observacao(embalagem, motivo):
    prompt = f'Operário industrial registrar pequena observação, descarte de {embalagem} por {motivo}, em uma frase txt, se o for \'Outro\', invente. Seja conciso'
    
    # Inicializa o cliente da API
    client = OpenAI (
        base_url = 'https://router.huggingface.co/v1',
        api_key = HUGGING_FACE
    )
    
    # Envia o prompt ao modelo e configura os parâmetros da geração
    completion = client.chat.completions.create (
        model = 'moonshotai/Kimi-K2-Instruct-0905',
        messages = [
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature = 0.2,
        max_tokens = 30
    )
    
    return completion.choices[0].message.content

In [14]:
# Pegando último ID
def ultimo_id(conn):
    sql = f'SELECT ID FROM DESCARTE ORDER BY ID DESC'
    res = conn.execute(text(sql))
    
    return res.scalar() + 1 # Pega somente primeira linha

In [15]:
# Pegando IDs das tabelas 
def pega_id(conn, tabela, tem_nome=False):
    if tem_nome:
        sql = f'SELECT ID, NOME FROM {tabela} ORDER BY ID'
    else:
        sql = f'SELECT ID FROM {tabela} ORDER BY ID'
    
    saida = conn.execute(text(sql))
    resultado = []
    
    if tem_nome:
        for s in saida:
            item = {'id': s[0], 'nome': s[1]}
            resultado.append(item)
        return resultado
    else:
        for s in saida:
            item = {'id': s[0]}
            resultado.append(item)
        return resultado

def pega_id_empregado(conn):
    sql = f'SELECT ID FROM EMPREGADO WHERE TIPO = \'OPERÁRIO\' ORDER BY ID '
    saida = conn.execute(text(sql))
    
    resultado = []
    
    for s in saida:
        item = {'id': s[0]}
        resultado.append(item)
    
    return resultado

def pega_id_embalagem(conn):
    sql = 'SELECT ID, NOME, VALOR_KG FROM EMBALAGEM ORDER BY ID'
    saida = conn.execute(text(sql))
    
    resultado = []
    for s in saida:
        item = {'id': s[0], 'nome': s[1], 'valor_kg': s[2]}
        resultado.append(item)
    return resultado

In [16]:
intervalo = {}
with engine.connect() as conn:
    intervalo['EMBALAGEM'] = pega_id_embalagem(conn)
    intervalo['MIDIA'] = pega_id(conn, 'MIDIA')
    intervalo['MOTIVO'] = pega_id(conn, 'MOTIVO', True)
    intervalo['EMPREGADO'] = pega_id_empregado(conn)

In [17]:
df_cod_motivo = pd.DataFrame(intervalo['MOTIVO'])
df_cod_embalagem = pd.DataFrame(intervalo['EMBALAGEM'])
df_cod_empregado = pd.DataFrame(intervalo['EMPREGADO'])
df_cod_midia = pd.DataFrame(intervalo['MIDIA'])

In [18]:
# Gerando informações de descarte

descartes = []
id = ultimo_id(engine.connect()) or 0

for i in range(120):
    descarte_kg = round(uniform(0.5, 15.0), 2)
    cod_motivo = choice(df_cod_motivo['id'])
    cod_embalagem = choice(df_cod_embalagem['id'])
    cod_empregado = choice(df_cod_empregado['id'])

    if df_cod_midia.empty:
        cod_midia = None
    else:
        cod_midia = choice(df_cod_midia['id'])
    
    nome_motivo = df_cod_motivo.loc[df_cod_motivo['id'] == cod_motivo, 'nome'].iloc[0]
    nome_embalagem = df_cod_embalagem.loc[df_cod_embalagem['id'] == cod_embalagem, 'nome'].iloc[0]
    valor_kg = float(df_cod_embalagem.loc[df_cod_embalagem['id'] == cod_embalagem, 'valor_kg'].iloc[0])

    descartes.append({
        "id": id,
        "descarte_kg": descarte_kg,
        "custo_estimado": round((descarte_kg * valor_kg), 2),
        "horario": f.date_time_this_year(),
        "observacao": None,
        "cod_embalagem": cod_embalagem,
        "cod_motivo": cod_motivo,
        "cod_empregado": cod_empregado,
        "cod_midia": choice([cod_midia, None])
    })
    
    id += 1

df_descartes = pd.DataFrame(descartes)
display(df_descartes)

,id,descarte_kg,custo_estimado,horario,observacao,cod_embalagem,cod_motivo,cod_empregado,cod_midia
0,4,5.50,2.64,2025-02-19 13:36:15,None,2,2,1,3.0
1,5,9.41,11.29,2025-03-14 21:42:44,None,1,10,4,NaN
2,6,7.53,24.10,2025-01-28 13:58:29,None,5,4,3,NaN
3,7,2.21,9.06,2025-01-18 07:07:56,None,4,3,2,2.0
4,8,14.02,57.48,2025-01-01 09:30:56,None,4,5,1,NaN
...,...,...,...,...,...,...,...,...,...
115,119,3.78,43.09,2025-06-11 00:41:51,None,3,4,4,2.0
116,120,10.07,114.80,2025-01-03 11:50:05,None,3,10,2,2.0
117,121,11.25,46.12,2025-01-03 08:21:04,None,4,8,3,1.0
118,122,10.90,44.69,2025-08-27 04:19:18,None,4,8,3,3.0


In [19]:
# Inserindo no banco de dados
df_descartes.to_sql('descarte', engine, if_exists='append', index=False)

120